In [1]:

from __future__ import annotations

import json
import re
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from xml.sax.saxutils import escape

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import (
    PageBreak,
    Paragraph,
    Preformatted,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)

# ============================================================
# RecoMart - Personalized Final Submission PDF Generator
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\barath\recomart-pipeline")
RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
REMEDIATED_RAW_ROOT = PROJECT_ROOT / "data" / "remediated" / "raw"
REMEDIATED_BRONZE_ROOT = PROJECT_ROOT / "data" / "remediated" / "bronze"
REPORT_ROOT = PROJECT_ROOT / "reports" / "validation"
LOG_ROOT = PROJECT_ROOT / "logs"
RUNS_ROOT = PROJECT_ROOT / "runs" / "prefect"
MLRUNS_ROOT = PROJECT_ROOT / "mlruns"
FEATURE_STORE_ROOT = PROJECT_ROOT / "feature_store"
SRC_ROOT = PROJECT_ROOT / "src"

OUTPUT_DIR = PROJECT_ROOT / "reports" / "submission"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PDF = OUTPUT_DIR / "RecoMart_Final_Submission_Barath.pdf"

# Exact validation evidence observed in uploaded files
VALIDATION_RUN_ID = "20260428T144031Z"
VALIDATION_INITIAL_STATUS = "FAIL"
VALIDATION_FINAL_STATUS = "PASS"

DISCOVERED_FILES = {
    "events_csv": RAW_ROOT / "retailrocket" / "load_date=2026-04-28" / "load_hour=14" / "events.csv",
    "category_tree_csv": RAW_ROOT / "retailrocket" / "load_date=2026-04-26" / "load_hour=19" / "category_tree.csv",
    "item_properties_part1_csv": RAW_ROOT / "retailrocket" / "load_date=2026-04-28" / "load_hour=14" / "item_properties_part1.csv",
    "item_properties_part2_csv": RAW_ROOT / "retailrocket" / "load_date=2026-04-28" / "load_hour=14" / "item_properties_part2.csv",
    "products_raw_json": RAW_ROOT / "dummyjson" / "load_date=2026-04-28" / "load_hour=14" / "products_raw.json",
    "categories_raw_json": RAW_ROOT / "dummyjson" / "load_date=2026-04-28" / "load_hour=14" / "categories_raw.json",
    "products_parquet": BRONZE_ROOT / "dummyjson" / "products.parquet",
    "categories_parquet": BRONZE_ROOT / "dummyjson" / "categories.parquet",
}

VALIDATION_ARTIFACTS = {
    "initial_issues": REPORT_ROOT / f"validation_issues_initial_{VALIDATION_RUN_ID}.csv",
    "initial_summary": REPORT_ROOT / f"validation_summary_initial_{VALIDATION_RUN_ID}.csv",
    "revalidation_issues": REPORT_ROOT / f"validation_issues_revalidated_{VALIDATION_RUN_ID}.csv",
    "revalidation_summary": REPORT_ROOT / f"validation_summary_revalidated_{VALIDATION_RUN_ID}.csv",
    "fix_log": REPORT_ROOT / f"fix_log_{VALIDATION_RUN_ID}.csv",
    "json_report": REPORT_ROOT / f"data_quality_report_{VALIDATION_RUN_ID}.json",
    "pdf_report": REPORT_ROOT / f"data_quality_report_{VALIDATION_RUN_ID}.pdf",
    "validation_log": LOG_ROOT / f"validation_log_{VALIDATION_RUN_ID}.jsonl",
}

SUBMISSION = {
    "student_name": "Barath",
    "roll_number": "__________",
    "course_name": "Data Management for Machine Learning",
    "assignment_title": "End-to-End Data Management Pipeline for a Recommendation System",
    "client_name": "RecoMart",
    "submission_date": datetime.now().strftime("%Y-%m-%d"),
    "business_problem": (
        "RecoMart requires a scalable, maintainable, and automated data management pipeline "
        "to ingest fresh multi-source e-commerce data, validate and remediate quality issues, "
        "prepare and transform datasets, manage reusable recommendation features, and train or "
        "refresh recommendation models with orchestration and experiment tracking."
    ),
    "objectives": [
        "Ingest at least two data types: CSV-based interaction/catalog files and API/raw JSON product data.",
        "Store source data in a structured local data lake partitioned by source and timestamp.",
        "Validate data quality, produce issue logs, auto-remediate failed datasets, and revalidate.",
        "Prepare clean datasets for EDA and downstream recommendation modeling.",
        "Implement feature engineering and a custom metadata-based feature store with versioned retrieval.",
        "Track model runs, parameters, and metrics using MLflow or equivalent metadata tracking.",
        "Orchestrate the full workflow end to end using Prefect.",
    ],
    "evaluation_metrics": ["Precision@K", "Recall@K", "NDCG@K"],
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
styles = getSampleStyleSheet()
styles.add(
    ParagraphStyle(
        name="SectionBody",
        parent=styles["BodyText"],
        fontSize=10,
        leading=14,
        spaceAfter=6,
    )
)
styles.add(
    ParagraphStyle(
        name="SmallCode",
        fontName="Courier",
        fontSize=7,
        leading=8,
        spaceAfter=6,
    )
)

def p(text: str, style: str = "SectionBody") -> Paragraph:
    return Paragraph(escape(text).replace("\n", "<br/>"), styles[style])

def bullets(items: List[str]) -> List[Any]:
    out: List[Any] = []
    for item in items:
        out.append(Paragraph(f"• {escape(str(item))}", styles["SectionBody"]))
    out.append(Spacer(1, 6))
    return out

def add_table(story: List[Any], title: str, rows: List[List[str]], col_widths=None) -> None:
    story.append(Paragraph(title, styles["Heading3"]))
    table = Table(rows, repeatRows=1, colWidths=col_widths)
    table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAD3")),
                ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
                ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
                ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                ("FONTSIZE", (0, 0), (-1, -1), 8),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("LEFTPADDING", (0, 0), (-1, -1), 4),
                ("RIGHTPADDING", (0, 0), (-1, -1), 4),
                ("TOPPADDING", (0, 0), (-1, -1), 4),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
            ]
        )
    )
    story.append(table)
    story.append(Spacer(1, 10))

def safe_read_text(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception as exc:
        return f"[Could not read {path}: {exc}]"

def parse_notebook_code(path: Path) -> str:
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        cells = []
        for cell in payload.get("cells", []):
            if cell.get("cell_type") == "code":
                src = "".join(cell.get("source", []))
                if src.strip():
                    cells.append(src)
        return "\n\n# ---- next code cell ----\n\n".join(cells) if cells else "[No code cells found]"
    except Exception as exc:
        return f"[Could not parse notebook {path}: {exc}]"

def gather_code_files() -> List[Path]:
    allowed = {".py", ".ipynb", ".yaml", ".yml", ".md", ".txt"}
    files: List[Path] = []
    if SRC_ROOT.exists():
        for path in SRC_ROOT.rglob("*"):
            if path.is_file() and path.suffix.lower() in allowed:
                files.append(path)

    # Add feature store configs if present
    if FEATURE_STORE_ROOT.exists():
        for path in FEATURE_STORE_ROOT.rglob("*"):
            if path.is_file() and path.suffix.lower() in allowed:
                files.append(path)

    return sorted(set(files))

def latest_prefect_run() -> Dict[str, str]:
    result = {
        "flow_name": "Not found",
        "latest_run_folder": "Not found",
        "executed_notebooks_folder": "Not found",
        "status_note": "Prefect run metadata could not be inferred.",
    }

    if not RUNS_ROOT.exists():
        result["status_note"] = "runs/prefect folder not found."
        return result

    run_dirs = [p for p in RUNS_ROOT.iterdir() if p.is_dir()]
    if not run_dirs:
        result["status_note"] = "No Prefect run folders found."
        return result

    latest = sorted(run_dirs, key=lambda p: p.name)[-1]
    result["latest_run_folder"] = str(latest)

    executed = latest / "executed_notebooks"
    if executed.exists():
        result["executed_notebooks_folder"] = str(executed)
        result["flow_name"] = latest.name
        result["status_note"] = "Latest Prefect run folder discovered successfully."
    else:
        result["flow_name"] = latest.name
        result["status_note"] = "Latest Prefect run found, but executed_notebooks folder missing."

    return result

def read_mlflow_meta_yaml(meta_file: Path) -> Dict[str, str]:
    data: Dict[str, str] = {}
    text = safe_read_text(meta_file)
    for line in text.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            data[k.strip()] = v.strip().strip("'").strip('"')
    return data

def try_parse_metric_file(metric_file: Path) -> Optional[float]:
    try:
        lines = [line.strip() for line in safe_read_text(metric_file).splitlines() if line.strip()]
        if not lines:
            return None
        last = lines[-1].split()
        return float(last[-1])
    except Exception:
        return None

def discover_mlflow_summary() -> Dict[str, Any]:
    summary = {
        "experiment_name": "Not found",
        "best_run_id": "Not found",
        "best_metric_name": "Not found",
        "best_metric_value": "Not found",
        "artifact_uri": "Not found",
        "status_note": "mlruns folder not found.",
    }

    if not MLRUNS_ROOT.exists():
        return summary

    experiment_dirs = [
        p for p in MLRUNS_ROOT.iterdir()
        if p.is_dir() and p.name != ".trash" and re.fullmatch(r"\d+", p.name)
    ]
    if not experiment_dirs:
        summary["status_note"] = "No MLflow experiment folders found."
        return summary

    # pick latest modified experiment
    experiment = sorted(experiment_dirs, key=lambda p: p.stat().st_mtime)[-1]
    exp_meta = experiment / "meta.yaml"
    if exp_meta.exists():
        exp_info = read_mlflow_meta_yaml(exp_meta)
        summary["experiment_name"] = exp_info.get("name", experiment.name)

    best_run = None
    best_metric_name = None
    best_metric_value = None
    best_artifact_uri = None

    for run_dir in experiment.iterdir():
        if not run_dir.is_dir():
            continue
        if run_dir.name in {"meta.yaml", "tags"}:
            continue

        meta_yaml = run_dir / "meta.yaml"
        metrics_dir = run_dir / "metrics"

        artifact_uri = "Not found"
        if meta_yaml.exists():
            run_info = read_mlflow_meta_yaml(meta_yaml)
            artifact_uri = run_info.get("artifact_uri", "Not found")

        if metrics_dir.exists():
            for metric_file in metrics_dir.iterdir():
                if metric_file.is_file():
                    value = try_parse_metric_file(metric_file)
                    if value is None:
                        continue
                    if best_metric_value is None or value > best_metric_value:
                        best_metric_value = value
                        best_metric_name = metric_file.name
                        best_run = run_dir.name
                        best_artifact_uri = artifact_uri

    if best_run:
        summary["best_run_id"] = best_run
        summary["best_metric_name"] = best_metric_name or "Not found"
        summary["best_metric_value"] = best_metric_value
        summary["artifact_uri"] = best_artifact_uri or "Not found"
        summary["status_note"] = "MLflow run and metric discovered from local mlruns folder."
    else:
        summary["status_note"] = "Experiment found, but no readable metric files were discovered."

    return summary

def infer_stage_folders() -> List[str]:
    if not SRC_ROOT.exists():
        return []
    return [p.name for p in sorted(SRC_ROOT.iterdir()) if p.is_dir()]

def repo_tree(root: Path, max_depth: int = 2) -> str:
    if not root.exists():
        return f"{root} [not found]"

    lines: List[str] = [root.name + "/"]

    def walk(path: Path, prefix: str, depth: int) -> None:
        if depth > max_depth:
            return
        children = sorted(path.iterdir(), key=lambda x: (x.is_file(), x.name.lower()))
        for child in children:
            lines.append(f"{prefix}{child.name}{'/' if child.is_dir() else ''}")
            if child.is_dir():
                walk(child, prefix + "    ", depth + 1)

    walk(root, "    ", 0)
    return "\n".join(lines)

def code_listing(path: Path) -> str:
    if path.suffix.lower() == ".ipynb":
        return parse_notebook_code(path)
    return safe_read_text(path)

# ------------------------------------------------------------
# Build report
# ------------------------------------------------------------
def build_story() -> List[Any]:
    prefect_info = latest_prefect_run()
    mlflow_info = discover_mlflow_summary()
    stage_folders = infer_stage_folders()
    code_files = gather_code_files()

    story: List[Any] = []

    # Title
    story.append(Paragraph("RecoMart Final Assignment Submission Report", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(p(f"Student Name: {SUBMISSION['student_name']}"))
    story.append(p(f"Roll Number: {SUBMISSION['roll_number']}"))
    story.append(p(f"Course: {SUBMISSION['course_name']}"))
    story.append(p(f"Assignment Title: {SUBMISSION['assignment_title']}"))
    story.append(p(f"Client: {SUBMISSION['client_name']}"))
    story.append(p(f"Project Root: {PROJECT_ROOT}"))
    story.append(p(f"Submission Date: {SUBMISSION['submission_date']}"))
    story.append(Spacer(1, 12))

    story.append(Paragraph("Executive Summary", styles["Heading2"]))
    story.append(
        p(
            "This submission presents an end-to-end recommendation data pipeline for RecoMart. "
            "The implementation covers problem formulation, multi-source ingestion, raw data storage, "
            "data validation with remediation and revalidation, preparation, feature engineering, "
            "a custom feature store, model tracking, and pipeline orchestration."
        )
    )
    story.extend(bullets(SUBMISSION["objectives"]))

    # 1
    story.append(Paragraph("1. Problem Formulation", styles["Heading2"]))
    story.append(p(SUBMISSION["business_problem"]))
    story.append(Paragraph("Expected Outputs", styles["Heading3"]))
    story.extend(
        bullets(
            [
                "Clean datasets for EDA.",
                "Engineered features for collaborative/content-based recommendation models.",
                "Deployable recommendation model artifacts and inference-ready datasets.",
                "Operational logs, validation reports, and tracked training metadata.",
            ]
        )
    )
    story.append(Paragraph("Evaluation Metrics", styles["Heading3"]))
    story.extend(bullets(SUBMISSION["evaluation_metrics"]))

    # 2
    story.append(Paragraph("2. Data Collection and Ingestion", styles["Heading2"]))
    story.append(
        p(
            "The pipeline ingests at least two data types, satisfying the assignment requirement: "
            "CSV-based RetailRocket interaction/catalog data and JSON-based DummyJSON product/category data."
        )
    )
    ingestion_rows = [["Source Key", "Exact Path", "Exists"]]
    for k, v in DISCOVERED_FILES.items():
        ingestion_rows.append([k, str(v), "Yes" if v.exists() else "No"])
    add_table(story, "Discovered Ingestion Files", ingestion_rows, col_widths=[4*cm, 10*cm, 2*cm])

    # 3
    story.append(Paragraph("3. Raw Data Storage", styles["Heading2"]))
    storage_rows = [
        ["Folder", "Purpose", "Exists"],
        [str(RAW_ROOT), "Raw source data", "Yes" if RAW_ROOT.exists() else "No"],
        [str(BRONZE_ROOT), "Structured bronze datasets", "Yes" if BRONZE_ROOT.exists() else "No"],
        [str(REMEDIATED_RAW_ROOT), "Remediated raw outputs", "Yes" if REMEDIATED_RAW_ROOT.exists() else "No"],
        [str(REMEDIATED_BRONZE_ROOT), "Remediated bronze outputs", "Yes" if REMEDIATED_BRONZE_ROOT.exists() else "No"],
        [str(REPORT_ROOT), "Validation reports", "Yes" if REPORT_ROOT.exists() else "No"],
        [str(LOG_ROOT), "Execution logs", "Yes" if LOG_ROOT.exists() else "No"],
    ]
    add_table(story, "Exact Storage Layout", storage_rows, col_widths=[8*cm, 6*cm, 2*cm])

    # 4
    story.append(Paragraph("4. Data Profiling and Validation", styles["Heading2"]))
    story.append(
        p(
            "The validation stage includes robust project-root discovery, issue logging, remediation, "
            "revalidation, and PDF/JSON/CSV report generation."
        )
    )
    validation_rows = [
        ["Field", "Value"],
        ["Validation Run ID", VALIDATION_RUN_ID],
        ["Initial Status", VALIDATION_INITIAL_STATUS],
        ["Final Status", VALIDATION_FINAL_STATUS],
        ["JSON Report", str(VALIDATION_ARTIFACTS["json_report"])],
        ["PDF Report", str(VALIDATION_ARTIFACTS["pdf_report"])],
        ["Fix Log", str(VALIDATION_ARTIFACTS["fix_log"])],
    ]
    add_table(story, "Validation Run Summary", validation_rows, col_widths=[5*cm, 11*cm])

    artifact_rows = [["Artifact", "Path", "Exists"]]
    for k, v in VALIDATION_ARTIFACTS.items():
        artifact_rows.append([k, str(v), "Yes" if v.exists() else "No"])
    add_table(story, "Validation Artifacts", artifact_rows, col_widths=[4*cm, 10*cm, 2*cm])

    # 5
    story.append(Paragraph("5. Data Preparation", styles["Heading2"]))
    story.append(
        p(
            "Prepared datasets are created after cleaning and remediation. Based on the validation pipeline, "
            "this includes fixing problematic raw and bronze files and carrying forward standardized datasets "
            "for downstream use."
        )
    )
    story.extend(
        bullets(
            [
                "Missing/invalid values addressed during remediation.",
                "Dataset consistency rechecked in revalidation.",
                "Prepared outputs routed into remediated/raw and remediated/bronze folders.",
            ]
        )
    )

    # 6
    story.append(Paragraph("6. Feature Engineering and Transformation", styles["Heading2"]))
    story.append(
        p(
            "Feature engineering logic is expected to transform prepared recommendation data into reusable "
            "training and inference features. The repository structure indicates a dedicated feature stage "
            "under src and a separate feature_store area for metadata-based retrieval."
        )
    )
    story.extend(
        bullets(
            [
                "Recommendation-oriented transformation stage exists under src.",
                "Feature outputs are intended to be version-aware and retrievable for training/inference.",
                "Supporting code is included in the appendix automatically from src/ and feature_store/.",
            ]
        )
    )

    # 7
    story.append(Paragraph("7. Feature Store", styles["Heading2"]))
    feature_registry = FEATURE_STORE_ROOT / "registry" / "feature_registry.yaml"
    feature_rows = [
        ["Item", "Value"],
        ["Feature Store Root", str(FEATURE_STORE_ROOT)],
        ["Registry File", str(feature_registry)],
        ["Registry Present", "Yes" if feature_registry.exists() else "No"],
    ]
    add_table(story, "Feature Store Metadata", feature_rows, col_widths=[4*cm, 12*cm])

    # 8
    story.append(Paragraph("8. Data Versioning and Lineage", styles["Heading2"]))
    story.extend(
        bullets(
            [
                "Raw data is partitioned by source plus timestamp folders such as load_date and load_hour.",
                "Validation outputs are versioned by run ID in reports/validation.",
                "Logs are timestamped and stored separately for auditability.",
                "Remediated outputs preserve lineage from original failed datasets to corrected datasets.",
            ]
        )
    )

    # 9
    story.append(Paragraph("9. Model Training and Evaluation", styles["Heading2"]))
    story.append(
        p(
            "The assignment requires tracked model metadata, parameters, and metrics. "
            "This report attempts to auto-discover those details from the local mlruns directory."
        )
    )
    mlflow_rows = [["Field", "Value"]]
    for k, v in mlflow_info.items():
        mlflow_rows.append([k, str(v)])
    add_table(story, "MLflow Discovery Summary", mlflow_rows, col_widths=[5*cm, 11*cm])

    # 10
    story.append(Paragraph("10. Pipeline Orchestration", styles["Heading2"]))
    story.append(
        p(
            "The assignment requires orchestration through Prefect, Airflow, or Dagster. "
            "This report auto-discovers the latest Prefect execution folder under runs/prefect."
        )
    )
    prefect_rows = [["Field", "Value"]]
    for k, v in prefect_info.items():
        prefect_rows.append([k, str(v)])
    add_table(story, "Prefect Discovery Summary", prefect_rows, col_widths=[5*cm, 11*cm])

    # 11
    story.append(Paragraph("11. Code Organization", styles["Heading2"]))
    story.append(
        p(
            "The assignment asks for source code organized by pipeline stage. "
            "The script below auto-lists current stage folders under src/."
        )
    )
    stage_rows = [["Detected Stage Folder"]]
    for s in stage_folders or ["No stage folders found under src/"]:
        stage_rows.append([s])
    add_table(story, "Detected src/ Stage Folders", stage_rows, col_widths=[16*cm])

    story.append(Paragraph("Repository Snapshot", styles["Heading3"]))
    story.append(Preformatted(repo_tree(SRC_ROOT, max_depth=2), styles["SmallCode"]))

    # 12
    story.append(PageBreak())
    story.append(Paragraph("12. Submission Checklist", styles["Heading2"]))
    story.extend(
        bullets(
            [
                "Problem definition, objectives, sources, and outputs documented.",
                "Ingestion evidence included for CSV and JSON sources.",
                "Structured raw/bronze/remediated/report/log folders documented.",
                "Validation report evidence included with exact run ID and artifacts.",
                "Feature store path and registry location documented.",
                "MLflow section included and auto-populated when mlruns is present.",
                "Prefect orchestration section included and auto-populated when runs/prefect is present.",
                "Code appendix included from src/ and feature_store/.",
                "Separate video walkthrough still required for final submission package.",
            ]
        )
    )

    # Appendix
    story.append(PageBreak())
    story.append(Paragraph("Appendix A: Selected Code Listings", styles["Heading2"]))

    if not code_files:
        story.append(p("No code files discovered under src/ or feature_store/."))
    else:
        for file_path in code_files:
            story.append(Paragraph(f"Code: {file_path.relative_to(PROJECT_ROOT)}", styles["Heading3"]))
            content = code_listing(file_path)
            if len(content) > 20000:
                content = content[:20000] + "\n\n[TRUNCATED FOR PDF SIZE]"
            story.append(Preformatted(content, styles["SmallCode"]))
            story.append(Spacer(1, 10))

    # Closing note
    story.append(PageBreak())
    story.append(Paragraph("Appendix B: Final Remarks", styles["Heading2"]))
    story.append(
        p(
            "This personalized report uses exact project paths and the exact validation run evidence already "
            "captured in the repository. If MLflow or Prefect metadata folders are present locally, they will "
            "be pulled into the final PDF automatically. Before submission, fill in the roll number and verify "
            "that the generated PDF, source code, video walkthrough, and zip package are all included."
        )
    )

    return story

def build_pdf() -> Path:
    doc = SimpleDocTemplate(
        str(OUTPUT_PDF),
        pagesize=A4,
        rightMargin=1.5 * cm,
        leftMargin=1.5 * cm,
        topMargin=1.5 * cm,
        bottomMargin=1.5 * cm,
    )
    story = build_story()
    doc.build(story)
    return OUTPUT_PDF

if __name__ == "__main__":
    pdf_path = build_pdf()
    print(f"Final personalized submission PDF created: {pdf_path}")


Final personalized submission PDF created: C:\Users\barath\recomart-pipeline\reports\submission\RecoMart_Final_Submission_Barath.pdf
